# Testing and Reliability in LangGraph

**What you will build:** a small test suite for a graph — and the reliability primitives that keep it running in production. You already learned to test agents offline in **1.5b** (fake models, millisecond runs, no API key); those ideas carry straight over. What's *new* is what a graph gives you to test — **nodes, routers, reducers, and interrupt/resume** — plus three production knobs the framework owns: **`RetryPolicy`**, **timeouts**, and **idempotent** side effects.

This is not a `pytest` tutorial (1.5b covered that); it's the LangGraph-specific half. Like 1.5b, none of it needs an API key — a good graph test replaces the model with a deterministic double.

In [ ]:
# Setup — LangGraph, offline. No API key needed: these tests use fakes, not a real model.
!pip install -q "langgraph>=1.2,<2.0"

from typing_extensions import TypedDict
from typing import Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import AIMessage
print("Ready — no OpenRouter key required for this notebook.")

## A node is just a function

Every LangGraph node has the same shape: **state in, partial-state dict out.** That makes it the easiest thing in the graph to test — call it directly, no graph, no model, no checkpointer. If a node does real work (parse, score, format), test that work in isolation before you ever wire an edge.

In [ ]:
def classify_priority(state: dict) -> dict:
    """Pure logic, no LLM: high priority if the message shouts or mentions money."""
    text = state["text"]
    urgent = text.isupper() or any(w in text.lower() for w in ("refund", "charged", "payment"))
    return {"priority": "high" if urgent else "normal"}

# Tests are plain asserts here — wrap them in pytest exactly as 1.5b did.
assert classify_priority({"text": "I was charged twice"})["priority"] == "high"
assert classify_priority({"text": "how do I rename a folder?"})["priority"] == "normal"
assert classify_priority({"text": "HELP ME NOW"})["priority"] == "high"
print("node ok")

## Test the router, not just the nodes

Conditional edges are where graph bugs hide: a wrong branch sends a ticket to the wrong specialist. A router is also *just a function* — `state -> next-node-name` — so pin its behaviour with a table of crafted states. This is the highest-value LangGraph test: routing is logic you own, and it's fully deterministic.

In [ ]:
def route(state: dict) -> str:
    return {"billing": "billing_node", "tech": "tech_node"}.get(state["category"], "fallback_node")

assert route({"category": "billing"}) == "billing_node"
assert route({"category": "tech"})     == "tech_node"
assert route({"category": "unknown"})  == "fallback_node"   # the branch people forget to test
print("router ok")

## Replace the model with a deterministic double

Nodes that call the LLM are non-deterministic — useless in a test unless you substitute the model, exactly as 1.5b did with `TestModel`. The graph-friendly way is to make the model a **dependency you can inject**: write the node as a small factory that takes an `llm`, so tests pass a fake and production passes the real `ChatOpenAI`.

In [ ]:
class FakeChat:
    """A stand-in for ChatOpenAI: returns a canned reply, no network."""
    def __init__(self, reply): self.reply = reply
    def invoke(self, messages): return AIMessage(content=self.reply)

def make_summarize(llm):                       # inject the model
    def summarize(state: dict) -> dict:
        reply = llm.invoke([{"role": "user", "content": state["text"]}])
        return {"summary": reply.content}
    return summarize

summarize = make_summarize(FakeChat("A one-line summary."))     # in a test: the fake
assert summarize({"text": "a long passage ..."}) == {"summary": "A one-line summary."}
print("model-node ok")   # in production: make_summarize(ChatOpenAI(...))

## Verify reducers and final state

The wiring — fan-out, reducers, edges — deserves a test of its own: build the graph, `invoke` it with known inputs, and assert the *accumulated* state. With fakes in the nodes this runs offline and deterministically, so you're testing the topology (2.1's reducer), not the model.

In [ ]:
class S(TypedDict):
    findings: Annotated[list[str], operator.add]   # concurrent writers append

def a(state: S) -> dict: return {"findings": ["a"]}
def b(state: S) -> dict: return {"findings": ["b"]}

g = StateGraph(S)
g.add_node("a", a); g.add_node("b", b)
g.add_edge(START, "a"); g.add_edge(START, "b")
g.add_edge("a", END);   g.add_edge("b", END)
final = g.compile().invoke({"findings": []})

assert sorted(final["findings"]) == ["a", "b"]    # the reducer merged both branches
print("reducer ok:", final["findings"])

## Test interrupt and resume

The pause-and-resume you built in 2.4 is behaviour to test *deliberately* — both that the graph stops before the irreversible step, and that a "no" is honoured. With an in-memory checkpointer the whole cycle runs offline, no human and no API key.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

SENT = []                                    # stand-in for a real "send email"
class HITL(TypedDict):
    draft: str
    sent: bool

def review(state: HITL) -> dict:
    decision = interrupt({"draft": state["draft"]})   # pause for a human (2.4)
    return {"sent": decision == "yes"}

def send(state: HITL) -> dict:
    if state["sent"]:
        SENT.append(state["draft"])
    return {}

gb = StateGraph(HITL)
gb.add_node("review", review); gb.add_node("send", send)
gb.add_edge(START, "review"); gb.add_edge("review", "send"); gb.add_edge("send", END)
graph = gb.compile(checkpointer=InMemorySaver())      # interrupt REQUIRES a checkpointer

cfg = {"configurable": {"thread_id": "t1"}}
paused = graph.invoke({"draft": "Hi there", "sent": False}, config=cfg)
assert "__interrupt__" in paused and SENT == []          # stopped; nothing sent yet

resumed = graph.invoke(Command(resume="yes"), config=cfg)
assert resumed["sent"] is True and SENT == ["Hi there"]  # approval honoured, sent once
print("interrupt/resume ok")

# The "no" path on a fresh thread — denial must be honoured too:
before = list(SENT)
graph.invoke({"draft": "Second draft", "sent": False}, config={"configurable": {"thread_id": "t2"}})
denied = graph.invoke(Command(resume="no"), config={"configurable": {"thread_id": "t2"}})
assert denied["sent"] is False and SENT == before        # denial recorded, nothing sent
print("denial path ok")

## Reliability: retry transient failures

A node that calls a flaky API shouldn't kill the run on the first hiccup. Attach a `RetryPolicy` and LangGraph re-runs the node on failure — no `try/except` in your logic. This is for *transient* faults (a dropped connection, a 503); permanent errors (a bad argument) should fail fast, which is why you scope `retry_on` to the exceptions actually worth retrying.

In [ ]:
from langgraph.types import RetryPolicy

class R(TypedDict):
    ok: bool

attempts = {"n": 0}
def flaky(state: R) -> dict:
    attempts["n"] += 1
    if attempts["n"] < 3:
        raise ConnectionError("transient")   # the first two calls fail
    return {"ok": True}

rg = StateGraph(R)
rg.add_node("flaky", flaky, retry_policy=RetryPolicy(max_attempts=3, retry_on=(ConnectionError,)))
rg.add_edge(START, "flaky"); rg.add_edge("flaky", END)
out = rg.compile().invoke({"ok": False})

assert out["ok"] is True and attempts["n"] == 3   # recovered on the third try
print("retry ok after", attempts["n"], "attempts")

## Reliability: bound a slow node with a timeout

Retries handle a node that *fails*; a timeout handles one that *hangs*. Since **langgraph>=1.2** you can bound a node's wall-clock time with `timeout=` — a number of seconds, or a `TimeoutPolicy(run_timeout=..., idle_timeout=...)` (both from `langgraph.types`) when you want to separate a total budget from an inactivity budget. When the limit is hit LangGraph raises **`NodeTimeoutError`** (`langgraph.errors`), and it applies **per attempt**, so it composes with `RetryPolicy`: each retry gets a fresh timeout. One caveat: this cancels **async** nodes — a synchronous node that blocks can't be interrupted mid-call, so keep I/O-bound work `async`.

In [ ]:
import asyncio
from langgraph.types import RetryPolicy, TimeoutPolicy   # TimeoutPolicy: the richer run/idle form
from langgraph.errors import NodeTimeoutError

class T(TypedDict):
    done: bool

async def slow(state: T) -> dict:
    await asyncio.sleep(5)                 # a hung dependency
    return {"done": True}

tg = StateGraph(T)
# 0.1s PER ATTEMPT; RetryPolicy gives it 2 tries, each with its own fresh timeout:
tg.add_node("slow", slow, timeout=0.1,
            retry_policy=RetryPolicy(max_attempts=2, retry_on=(NodeTimeoutError,)))
tg.add_edge(START, "slow"); tg.add_edge("slow", END)

try:
    await tg.compile().ainvoke({"done": False})
except NodeTimeoutError:
    print("timed out on each attempt, then gave up — as expected")
# TimeoutPolicy(run_timeout=..., idle_timeout=...) is the same idea with an added inactivity bound.

## Idempotency: the catch with retries and resume

There's a sting in the tail. Both `RetryPolicy` *and* interrupt/resume can **re-run a node** — so a node that performs an external side effect (charge a card, send an email) can fire it twice. 2.4 warned about exactly this; here's the fix. Make the side effect **idempotent**: guard it with a key so a second run is a no-op.

In [ ]:
CHARGED = []                       # pretend external ledger

def charge_bad(state: dict) -> dict:
    CHARGED.append(state["order"])        # keyless: re-running DOUBLE-charges
    return {"done": True}

def charge_good(state: dict) -> dict:
    op = f"refund:{state['order']}"       # stable, operation-specific key (an order has many ops)
    if op not in CHARGED:                 # teaching stand-in for an ATOMIC check — see the note
        CHARGED.append(op)
    return {"done": True}

# Simulate a retry/replay by running the node twice on the same order:
CHARGED.clear(); charge_bad({"order": "A-1"});  charge_bad({"order": "A-1"})
assert CHARGED == ["A-1", "A-1"]              # charged twice — the bug

CHARGED.clear(); charge_good({"order": "A-1"}); charge_good({"order": "A-1"})
assert CHARGED == ["refund:A-1"]             # charged once — correct
print("idempotency ok")

The lesson isn't "avoid retries" — it's that **retries and resume make re-execution normal, so every node that touches the outside world must be safe to run more than once.** Reducers append, checkpoints replay, policies retry.

Two caveats on the toy above. First, key each side effect on a **stable, operation-specific** value (`refund:A-1`), not just the order id — the same order may legitimately have several different operations (a charge, a partial refund, a note) that must not collide. Second, an in-memory `if key not in list` is only a teaching stand-in: it isn't atomic, so two concurrent runs can both pass the check. Real idempotency uses an **atomic** mechanism — a provider-supported idempotency key, a database unique constraint, an upsert, or a transactionally stored operation key. This is the graph-side twin of the idempotency keys from 3.0's double-texting note.

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Test a missing branch.** Add a `refund` category to `route` and write the `assert` *first* (watch it fail), then make it pass. **Done when:** the new branch has a test and the fallback still works.
2. **Fake a tool call.** Extend `FakeChat` to return an `AIMessage` with `tool_calls=[...]`, and test a node that branches on whether the model asked for a tool. **Done when:** your assert proves the tool path with no network call.
3. **See a retry duplicate a side effect.** Put a node in a real graph with `RetryPolicy(max_attempts=2)` that **first** appends to an external list, **then** raises a transient error on its first run — so the retry runs the whole node again. **Done when:** the list has two entries, and you can fix it by keying the append so the second run is a no-op.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 —**

```python
def route(state):
    table = {"billing": "billing_node", "tech": "tech_node", "refund": "refund_node"}
    return table.get(state["category"], "fallback_node")

assert route({"category": "refund"}) == "refund_node"
assert route({"category": "unknown"}) == "fallback_node"   # fallback still holds
```

**2 —** A fake that emits a tool call, then a node that reads it:

```python
tool_msg = AIMessage(content="", tool_calls=[{"name": "search", "args": {"q": "x"}, "id": "1"}])
def wants_tool(msg): return "tools" if msg.tool_calls else "end"
assert wants_tool(tool_msg) == "tools"
assert wants_tool(AIMessage(content="done")) == "end"
```

This is 2.2's `should_continue` router, tested without a model — the branch that decides *tool vs stop* is pure logic once the message exists.

**3 —** Append *then* fail, so the retry re-runs the whole node:

```python
LEDGER, tries = [], {"n": 0}
def charge(state):
    LEDGER.append(state["order"])          # (a) the side effect happens first
    tries["n"] += 1
    if tries["n"] == 1:
        raise ConnectionError("transient")  # (b) fail AFTER it — the retry re-runs the whole node
    return {"done": True}
# add_node('charge', charge, retry_policy=RetryPolicy(max_attempts=2, retry_on=(ConnectionError,)))
# after one invoke: LEDGER == ['A-1', 'A-1'] — the retry double-charged
```

The fix is the idempotent guard from the cell above: key the append (`op = f"refund:{state['order']}"`) so the second run is a no-op. The rule: **any node that can be retried or resumed must be idempotent** — and in production that guard must be *atomic* (a unique constraint / upsert), not an in-memory `if`.
::::

::::{dropdown} 🔧 Common issues
:color: info

| Symptom | Likely cause | Fix |
|---|---|---|
| `invoke` returns state with no `__interrupt__` | Compiled without a checkpointer | `interrupt` needs `compile(checkpointer=...)` (2.3); an interrupt with no checkpoint has nowhere to save |
| The retry never fires | Your exception isn't covered by `retry_on` | Widen `retry_on`, or drop it to use the default policy — but keep *permanent* errors out so they fail fast |
| Resuming re-runs an earlier node's side effect | That node isn't idempotent | Key the side effect (order id, request id) so a replay is a no-op — see the idempotency cell |
| A parallel node's write clobbers another's | The shared key has no reducer | Add `Annotated[list, operator.add]` (2.1); a single-writer key needs none |
| A timeout never fires on a slow node | The node is synchronous and blocks, so it can't be cancelled mid-call | Make I/O-bound nodes `async`; `timeout=` cancels async work, not a blocking sync call |
::::

**Block 2 complete.** You can build graphs (2.1), loop (2.2), persist (2.3), pause (2.4), and now **test and harden** them — nodes, routers, reducers and interrupt/resume under test, with `RetryPolicy`, timeouts, and idempotent side effects for production. Next, **Block 3** takes an agent to the outside world: deploy it behind an API (3.0), put a front end on it (3.1), and decide *which tool* each job deserves (3.3).